In [1]:
from tensorflow.keras.datasets import mnist
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
(X_train, y_train), (X_test, y_test) = mnist.load_data()

In [4]:
X_train

array([[[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       ...,

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0]],

       [[0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        [0, 0, 0, ..., 0, 0, 0],
        ...,
        [0, 0, 0, ..., 

In [144]:
X_train_flat = X_train.reshape(-1, 784) 
X_test_flat = X_test.reshape(-1, 784)
np.shape(X_test_flat)

##By passing -1, you are saying: "I know I want each row to have 784 elements, so look at the total size of my data and figure out 
# how many rows that makes."

(10000, 784)

In [145]:
##Transpose So that the formula Z = WX+b matches and we don't have to do W^T later.  

X_test_flat = X_test_flat.T
X_train_flat = X_train_flat.T

In [146]:
np.shape(X_test_flat)

(784, 10000)

In [147]:
X_test_flat_normalized = X_test_flat/255
X_train_flat_normalized = X_train_flat/255
X_test_flat_normalized

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

### Initial Weights


In [334]:
##xavier initialization of weights

sigma1 = np.sqrt(2/(784 + 128)) ## sigma = root(2/(input+output)) 

W1 = sigma1 * np.random.randn(128, 784) ##weight = sigma * normal distribution

b1 = np.zeros((128,1))

###
sigma2 = np.sqrt(2/(128+64))

W2 = sigma2 * np.random.randn(64, 128)

b2 = np.zeros((64,1))

###
sigma3 = np.sqrt(2/(64+10))

W3 = sigma3 * np.random.randn(10, 64)

b3 = np.zeros((10,1))



In [335]:
def Softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=0, keepdims=True))
    sum_exp = np.sum(exp_Z, axis=0, keepdims=True)

    return exp_Z / sum_exp

#### Forward pass (Guesses, here is my current prediction)


In [336]:
def forward_pass(X):
    ##layer 1
    Z1 = W1@X+b1 ##raw linear output

    #ReLU
    A1 = np.maximum(0, Z1) ##activation function

    ##layer 2

    Z2 = W2@A1 + b2

    A2 = np.maximum(0, Z2) 

    ##layer 3

    Z3 = W3@A2 + b3

    A3 = Softmax(Z3)

    return Z1, A1, Z2, A2, Z3, A3




In [347]:
Z1, A1, Z2, A2, Z3, A3 = forward_pass(X_train_flat_normalized)
np.shape(A3)

(10, 60000)

#### Loss Function 

In [338]:
def complete_loss(A3, y):
    
    m = len(y)
    y_correct = A3[y, range(m)]

    Loss = (-1/m)*(np.sum(np.log(y_correct))) #L = -1/m · Σ log(ŷ_correct)

    return Loss

In [339]:
Loss = complete_loss(A3, y_train)
Loss

np.float64(2.3361955774583874)

##### one hot encoding on the varification 

In [340]:
one_hot_y_train = pd.get_dummies(y_train, dtype=int).T.to_numpy()
one_hot_y_test = pd.get_dummies(y_test, dtype=int).T.to_numpy()

In [341]:
one_hot_y_train

array([[0, 1, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [0, 0, 0, ..., 0, 0, 0]])

### Backward Pass


In [342]:
def backward_pass(X, y, A3, Z2, A2, Z1, A1):

    m = y.shape[1]

    ##layer 3
    softmax_cross_entropy_gradient = A3-y #dL/dZ3

    dL_by_dW3 = (softmax_cross_entropy_gradient @ A2.T)/m ##weight

    dL_by_db3 = np.sum(softmax_cross_entropy_gradient, axis=1, keepdims=True) / m ##bias

    dL_by_dA2 = W3.T @ softmax_cross_entropy_gradient


    #layer 2
    dL_by_dZ2 = dL_by_dA2 * (Z2 > 0) ##dL/dZ2 = dL/dA2 * ReLU'(Z2)

    dL_by_dW2 = (dL_by_dZ2 @A1.T) /m

    dL_by_db2 = np.sum(dL_by_dZ2, axis=1, keepdims=True)/m

    dL_by_dA1 = W2.T @ dL_by_dZ2

    #layer 1

    dL_by_dZ1 = dL_by_dA1 * (Z1 > 0)

    dL_by_dW1 = (dL_by_dZ1 @ X.T)/m

    dL_by_db1 = np.sum(dL_by_dZ1, axis=1, keepdims=True) / m

    ##finally return
    return {
            "dW1": dL_by_dW1, 
            "db1": dL_by_db1, 
            "dW2": dL_by_dW2, 
            "db2": dL_by_db2, 
            "dW3": dL_by_dW3, 
            "db3": dL_by_db3
        }


#### Update function

In [343]:


def update_function(W, b, dW, db, r, mW, mb, vW, vb, t):

    # honey wake up, Sonics using Adams Optimizer!

    b1 = 0.9
    b2 = 0.999


    # Momentum

    mW = b1 * mW + (1-b1) * dW
    mb = b1 * mb + (1-b1) * db

    #Velocity
    vW = b2 * vW + (1-b2)*(dW**2)
    vb = b2 * vb + (1-b2)*(db**2)

    ##bias correction

    mW_hat = mW/(1-max(b1**t, 1e-8))
    mb_hat = mb/(1-max(b1**t, 1e-8))

    vW_hat = vW/(1-max(b2**t, 1e-8))
    vb_hat = vb/(1-max(b2**t, 1e-8))

    ##gradient descent

    W -= r * mW_hat / (np.sqrt(vW_hat) + 1e-8) 

    b -= r * mb_hat / (np.sqrt(vb_hat) + 1e-8) 

    return W, b,  mW, mb, vW, vb

In [344]:
def training_loop():

    global W1, b1, W2, b2, W3, b3

    epochs = 100

    t = 0

    learning_rate = 0.001

    batch_size = 32

    X_numbers = X_train_flat_normalized.shape[1]

    mW1 = np.zeros_like(W1)
    mb1 = np.zeros_like(b1)
    vW1 = np.zeros_like(W1)
    vb1 = np.zeros_like(b1)

    mW2 = np.zeros_like(W2)
    mb2 = np.zeros_like(b2)
    vW2 = np.zeros_like(W2)
    vb2 = np.zeros_like(b2)

    mW3 = np.zeros_like(W3)
    mb3 = np.zeros_like(b3)
    vW3 = np.zeros_like(W3)
    vb3 = np.zeros_like(b3)





    for x in range(epochs):

        index = np.random.permutation(X_numbers)

        X_shuffel = X_train_flat_normalized[:, index]
        Y_shuffel = one_hot_y_train[:, index]

        

        for i in range(0, X_numbers, batch_size):

            X_batch = X_shuffel[:, i:i+batch_size]
            y_batch = Y_shuffel[:, i:i+batch_size]

            Z1, A1, Z2, A2, Z3, A3 = forward_pass(X_batch)

            t +=1


            gradians = backward_pass(X_batch, y_batch, A3, Z2, A2, Z1, A1)

            W1, b1 ,mW1, mb1, vW1, vb1 = update_function(W1, b1, gradians["dW1"], gradians["db1"], learning_rate, mW1, mb1, vW1, vb1, t)
            W2, b2, mW2, mb2, vW2, vb2 = update_function(W2, b2, gradians["dW2"], gradians["db2"], learning_rate, mW2, mb2, vW2, vb2, t)
            W3, b3, mW3, mb3, vW3, vb3 = update_function(W3, b3, gradians["dW3"], gradians["db3"], learning_rate, mW3, mb3, vW3, vb3, t)

        if x % 10 == 0:
            _, _, _, _, _, A3_full = forward_pass(X_train_flat_normalized)
            print(f"epoch: {x}, loss: {complete_loss(A3_full, y_train)}")
                
            



In [345]:
training_loop()

epoch: 0, loss: 0.10864669471598613
epoch: 10, loss: 0.011896383502074292
epoch: 20, loss: 0.007349131972383387
epoch: 30, loss: 0.009987389716445756
epoch: 40, loss: 0.0017002596438177276
epoch: 50, loss: 0.0024334971522281795
epoch: 60, loss: 0.004693270564813834
epoch: 70, loss: 0.002417591392979183
epoch: 80, loss: 0.0004216496855011793
epoch: 90, loss: 0.003930454657210459


In [346]:
Z1, A1, Z2, A2, Z3, A3 = forward_pass(X_test_flat_normalized)
prediction = np.argmax(A3, axis=0)

accuracy = np.mean(prediction == y_test) * 100
print(f"accuracy: {accuracy}%")

accuracy: 97.97%


In [348]:
np.save('weights/W1.npy', W1)
np.save('weights/b1.npy', b1)
np.save('weights/W2.npy', W2)
np.save('weights/b2.npy', b2)
np.save('weights/W3.npy', W3)
np.save('weights/b3.npy', b3)